In [ ]:
# This is run from main working directory
import mlrun
mlrun.set_environment(api_path="http://localhost:30070")
project = mlrun.load_project(name="legalcontractextractor", context="../")
# print(project.to_yaml())

from dotenv import load_dotenv
load_dotenv()

import os
ENV = os.environ["ENV"]

# Run the workflow

In [7]:
from datetime import datetime
source_path = f"s3://{ENV}-mlops-bucket-haviv/data/raw"
version = datetime.now().strftime("%Y%m%d_%H%M")

output_path = f"s3://{ENV}-mlops-bucket-haviv/data/processed_training/"

In [12]:
# Train dataset
runobj = mlrun.run_function(
    "raw-proc",  # use the function name registered in the register_funcs.ipynb file
    inputs={"input_uri": f"{source_path}/train.parquet"},
    params={
        "label_column": "inference",
        "artifact_key": f"train_data",
        "version": version,
        "output_uri_path": "s3://legal-llama-data/processed_training",
    },
    local=True,
)

# Validation dataset
runobj = mlrun.run_function(
    "raw-proc",
    inputs={"input_uri": f"{source_path}/validation.parquet"},
    params={
        "label_column": "inference",
        "artifact_key": f"validation_data",
        "version": version,
        "output_uri_path": "s3://legal-llama-data/processed_training",
    },
    local=True,
)

# Test dataset
runobj = mlrun.run_function(
    "raw-proc",
    inputs={"input_uri": f"{source_path}/test.parquet"},
    params={
        "label_column": "inference",
        "artifact_key": f"test_data",
        "version": version,
        "output_uri_path": "s3://legal-llama-data/processed_training",
    },
    local=True,
)

> 2026-08-14 15:59:15,224 [info] Storing function: {"db":"http://localhost:30070","name":"raw-proc-process-raw","uid":"2b7d10dd5ee449189e592de4586fe4ee"}
s3://sand-mlops-bucket-haviv/data/raw/train.parquet processed and written to s3://legal-llama-data/processed_training/20260814_1551


project,uid,iter,start,end,state,kind,name,labels,inputs,parameters,results,artifact_uris
legalcontractextractor,...6fe4ee,0,Aug 14 07:59:15,NaT,completed,run,raw-proc-process-raw,kind=localowner=jerrohost=Nitro_53,input_uri,label_column=inferenceartifact_key=train_dataversion=20260814_1551output_uri_path=s3://legal-llama-data/processed_training,,train_data=store://datasets/legalcontractextractor/raw-proc-process-raw_train_data#0:20260814_1551@2b7d10dd5ee449189e592de4586fe4ee^999e9fcfcda06d64a02dd6f448ab641383b7b7c8


> 2026-08-14 15:59:25,454 [info] Run execution finished: {"name":"raw-proc-process-raw","status":"completed"}
> 2026-08-14 15:59:25,456 [info] Storing function: {"db":"http://localhost:30070","name":"raw-proc-process-raw","uid":"c212ce0ee84e40418d62ef43fad5a8ef"}
s3://sand-mlops-bucket-haviv/data/raw/validation.parquet processed and written to s3://legal-llama-data/processed_training/20260814_1551


project,uid,iter,start,end,state,kind,name,labels,inputs,parameters,results,artifact_uris
legalcontractextractor,...d5a8ef,0,Aug 14 07:59:25,NaT,completed,run,raw-proc-process-raw,kind=localowner=jerrohost=Nitro_53,input_uri,label_column=inferenceartifact_key=validation_dataversion=20260814_1551output_uri_path=s3://legal-llama-data/processed_training,,validation_data=store://datasets/legalcontractextractor/raw-proc-process-raw_validation_data#0:20260814_1551@c212ce0ee84e40418d62ef43fad5a8ef^a85f5e46a24a633a38110fe912b3c5759647623e


> 2026-08-14 15:59:29,313 [info] Run execution finished: {"name":"raw-proc-process-raw","status":"completed"}
> 2026-08-14 15:59:29,320 [info] Storing function: {"db":"http://localhost:30070","name":"raw-proc-process-raw","uid":"17d0da779dbc45e4a14fb4e3e25f45c9"}
s3://sand-mlops-bucket-haviv/data/raw/test.parquet processed and written to s3://legal-llama-data/processed_training/20260814_1551


project,uid,iter,start,end,state,kind,name,labels,inputs,parameters,results,artifact_uris
legalcontractextractor,...5f45c9,0,Aug 14 07:59:29,NaT,completed,run,raw-proc-process-raw,kind=localowner=jerrohost=Nitro_53,input_uri,label_column=inferenceartifact_key=test_dataversion=20260814_1551output_uri_path=s3://legal-llama-data/processed_training,,test_data=store://datasets/legalcontractextractor/raw-proc-process-raw_test_data#0:20260814_1551@17d0da779dbc45e4a14fb4e3e25f45c9^eaf7773615b3193a8d1249044faca6a249fdf43a


> 2026-08-14 15:59:33,185 [info] Run execution finished: {"name":"raw-proc-process-raw","status":"completed"}


In [ ]:
# For kubeflow pipelines: not using during development because of ghost runs and caching issues..for another time
# from datetime import datetime

# run_obj = project.run(
#     name="evaluate_noTrain",
#     arguments={
#         "source_path": "s3://legal-llama-data/raw",
#         "version": datetime.now().strftime("%Y%m%d_%H%M")
#     },
#     local=True,   # Run the pipeline sequence locally
#     watch=True    # Print the progress to the console
# )

## Validation

In [16]:
from IPython.display import display

# Testing direct access to S3 data (underlying s3fs)
# data_uri = "s3://legal-llama-data/raw/test.parquet"
# df = mlrun.get_dataitem(data_uri).as_df() #this reads into a dataframe and only works if the file is a csv/parquet... jsonl does not work
# display(df.head(1))

data_uri = "store://datasets/legalcontractextractor/raw-proc-process-raw_test_data:latest"
data_pointer = mlrun.get_dataitem(data_uri)
print(data_pointer.url) # S3 path

s3://legal-llama-data/processed_training/20260814_1551/raw-proc-process-raw/0/test_data.parquet


In [17]:
# Testing data versioning and access to registered datasets in MLRun
data_uri = "store://datasets/legalcontractextractor/raw-proc-process-raw_test_data:latest"
#data_uri = "store://datasets/finetune-legal-extractor/raw-proc-process-raw_test_data:20260427_1945"

# Fetch the item and immediately convert it to a Pandas DataFrame
df = mlrun.get_dataitem(data_uri).as_df()
print(df.head(1)['inference'][0])


[{'hypothesis': "Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information.", 'hypothesis_id': 'nda-11', 'label': 'not_mentioned', 'source_clause': ''}
 {'hypothesis': 'Receiving Party shall destroy or return some Confidential Information upon the termination of Agreement.', 'hypothesis_id': 'nda-16', 'label': 'entailment', 'source_clause': 'The Recipient shall immediately return and redeliver to the other all tangible material embodying the JEA Confidential Information provided hereunder and all notes, summaries, memoranda, drawings, manuals, records, excerpts or derivative information deriving there from and all other documents or materials ("Notes") (and all copies of any of the foregoing, including "copies" that have been converted to computerized media in the form of image, data or word processing files either manually or by image capture) based on or including any JEA Confidential Information, in whatever form of storage or re

In [ ]:
"""
# I blame mlrun for this weird behaviour, instead of the URI on the UI with the ://files path segment, it uses ://datasets
# Now that we are not using 
artifact_uri = "store://datasets/finetune-legal-extractor/raw-proc-process-raw_test_data:latest"

data_item = mlrun.get_dataitem(artifact_uri)
s3_path = data_item.url
print(s3_path) 

import pandas as pd
import io

raw_bytes = data_item.get()
df = pd.read_json(io.BytesIO(raw_bytes), orient="records", lines=True)

inferences = df.head(1)['inference'][0]
for i in inferences:
    print(i)
"""


In [14]:
artifact = project.get_artifact(key="raw-proc-process-raw_validation_data", tag="20260814_1551") # use the db-key not key
# this wont work project.get_artifact(key="train_data")

print(artifact.get_store_url())
print(artifact.target_path) # points to the source path of the latest version of the artifact
print(artifact.db_key)
print(artifact.key)

store://datasets/legalcontractextractor/raw-proc-process-raw_validation_data#0:20260814_1551@c212ce0ee84e40418d62ef43fad5a8ef^a85f5e46a24a633a38110fe912b3c5759647623e
s3://legal-llama-data/processed_training/20260814_1551/raw-proc-process-raw/0/validation_data.parquet
raw-proc-process-raw_validation_data
validation_data


In [15]:
# all datasets
artifacts = project.list_artifacts()
datasets = [artifact for artifact in artifacts if artifact['kind'] == "dataset"]
for i in datasets:
    print(i['metadata']['key'], i['metadata']['tag'])

test_data latest
test_data 20260814_1551
validation_data latest
validation_data 20260814_1551
train_data latest
train_data 20260814_1551
train_data None


# Deleting artifacts directly

In [ ]:
adsasd

In [ ]:
# The only way to properly delete a data artifact and its historical versions
import os 

artifact = project.get_artifact(key="raw-proc-process-raw_train_data")
project.delete_artifact(artifact, 
                        deletion_strategy=mlrun.common.schemas.artifact.ArtifactsDeletionStrategies.data_force,
                        secrets={
                            "AWS_ACCESS_KEY_ID": os.environ['AWS_ACCESS_KEY_ID'],
                            "AWS_SECRET_ACCESS_KEY": os.environ['AWS_SECRET_ACCESS_KEY']
                            }
                        )


In [ ]:
# This deletes all artifacts in the database
db = mlrun.get_run_db()
db.del_artifacts(project=project.metadata.name)

print("all artifacts wiped from " + project.metadata.name)

In [ ]:
project.spec.get_code_path()